# ARD-VAE: Table 2 Replication on Kaggle (2x NVIDIA T4 GPU)
### Official Implementation of *"ARD-VAE: A Statistical Formulation to Find the Relevant Latent Dimensions of Variational Autoencoders"* (WACV 2025)
**Authors:** Surojit Saha, Sarang Joshi, Ross Whitaker (University of Utah)

---
### Notebook Objectives:
1. **Hardware Verification**: Verify Kaggle 2x Tesla T4 GPUs and environment.
2. **Setup Repository**: Clone and configure the Kaggle-ready ARD-VAE codebase.
3. **Reference Inception Statistics**: Pre-compute / prepare Inception statistics for FID evaluation.
4. **Train ARD-VAE on MNIST ($L=16$)**: Run 5 seeds (parallelized across both GPUs) to replicate paper Table 2.
5. **Train ARD-VAE on CIFAR-10 ($L=128$)**: Run 5 seeds (parallelized across both GPUs).
6. **Active Dimension Analysis**: Run Jacobian sensitivity weighting (Eq. 19-20) to extract active latent dimensions.
7. **Sample Generation & Quality Evaluation**: Generate samples from active dimensions, compute **FID**, **Precision**, and **Recall**.
8. **Table 2 Reproduction**: Output the full comparative table (Mean ± Std across seeds) vs. baseline models reported in the paper.
9. **Visual Verification**: Display sample reconstructions, generations, and cumulative explained variance curves.


## 1. Hardware & GPU Check
Verify that Kaggle has allocated 2x NVIDIA T4 GPUs with memory growth configured.


In [ ]:
!nvidia-smi

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"Detected {len(gpus)} GPU(s): {[gpu.name for gpu in gpus]}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)



## 2. Clone & Setup the Repository
Clone the modified ARD-VAE repository and navigate to its directory.


In [ ]:
import os

REPO_URL = "https://github.com/Param45/ARD-VAE.git"  # Replace with your repository URL if pushing to a custom repo
REPO_DIR = "/kaggle/working/ARD-VAE"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q scipy matplotlib imageio tqdm



## 3. Prepare Reference Inception Statistics for FID
Fréchet Inception Distance (FID) compares activations of generated samples against reference statistics ($\mu, \Sigma$) of real dataset images.
This step pre-computes and caches these statistics for MNIST and CIFAR-10 so FID evaluation runs smoothly.


In [ ]:
%cd {REPO_DIR}/eval/study_fid/compute_fid
# Run in a subprocess: an Inception session inside this kernel would hold GPU memory for the rest of the notebook.
# Saves mu, sigma and the raw Inception features (needed for precision/recall).
!python prepare_fid_stats.py --dataset ALL --out_dir fid_stats --samples 10000
%cd {REPO_DIR}

## 4. Multi-GPU Training Queue
All 10 training runs (5 CIFAR-10 seeds + 5 MNIST seeds) go through **one job queue over the 2x T4 GPUs**:
each GPU starts the next job as soon as it is free. The long CIFAR-10 jobs are queued first, so the short MNIST
jobs run on the GPU that would otherwise sit idle while CIFAR-10 seed 5 trains — MNIST adds no extra wall time.

Each run writes its full output to `train_logs/`; this cell prints a short progress line every few minutes.
Seeds whose final checkpoint already exists are **skipped**, so after a session restart you can simply re-run the cells.

In [ ]:
import subprocess
import time
import os
from config.local_config import configurations

def _tail(path, n=25):
    with open(path, errors='replace') as f:
        return ''.join(f.readlines()[-n:])

def _last_epoch_line(path):
    with open(path, errors='replace') as f:
        lines = [l.strip() for l in f if l.startswith('Epoch no') or l.startswith('Validation data reconstruction loss')]
    return ' | '.join(lines[-2:]) if lines else 'starting...'

def _start_next(gpu, pending, env):
    """Starts the next unfinished job from `pending` on `gpu`; returns (name, process, log) or None."""
    while pending:
        config_id, run_id = pending.pop(0)
        cfg = configurations[config_id]
        name = f"{cfg['dataset_name']} seed {run_id}"
        ckpt = f"logs/{cfg['dataset_name']}/Dim_{cfg['latent_dim']}/Run_{run_id}/Models/checkpoint"
        if os.path.exists(ckpt):
            print(f">>> {name}: final checkpoint exists, skipping ({ckpt})")
            continue
        log_path = f"train_logs/config{config_id}_run{run_id}.log"
        cmd = f"python Main.py --run_id {run_id} --config_id {config_id} --gpu {gpu}"
        print(f">>> Starting {name} on GPU {gpu}: {cmd}   (log: {log_path})", flush=True)
        proc = subprocess.Popen(cmd, shell=True, env=env, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
        return name, proc, log_path
    return None

def run_jobs(jobs, gpus=(0, 1), poll_seconds=300):
    """
    Runs (config_id, run_id) jobs on a queue: each GPU takes the next job as soon as it is free.
    Put the longest jobs first.
    """
    os.makedirs("train_logs", exist_ok=True)
    env = dict(os.environ, TQDM_DISABLE="1", PYTHONUNBUFFERED="1")  # readable logs, epoch lines written immediately
    pending, running, failed = list(jobs), {}, []
    start = last_report = time.time()
    while True:
        for gpu in gpus:
            if gpu not in running:
                job = _start_next(gpu, pending, env)
                if job:
                    running[gpu] = job
        if not running:
            break
        time.sleep(30)
        for gpu, (name, proc, log_path) in list(running.items()):
            if proc.poll() is None:
                continue
            del running[gpu]
            if proc.returncode == 0:
                print(f">>> {name} finished successfully! ({(time.time() - start) / 60:.1f} min elapsed)", flush=True)
            else:
                failed.append(name)
                print(f"!!! {name} FAILED (returncode {proc.returncode}). Last lines of {log_path}:\n{_tail(log_path)}", flush=True)
        if time.time() - last_report >= poll_seconds:
            last_report = time.time()
            for name, proc, log_path in running.values():
                print(f"[{(time.time() - start) / 60:6.1f} min] {name}: {_last_epoch_line(log_path)}", flush=True)

    print(f"\nAll jobs done in {(time.time() - start) / 60:.1f} min." + (f" FAILED: {failed}" if failed else ""))

## 5. Train ARD-VAE on CIFAR-10 ($L=128$) and MNIST ($L=16$) — Table 2 Setup
| | CIFAR-10 (config 4) | MNIST (config 3) |
|---|---|---|
| Latent dimension $L$ | $128$ | $16$ |
| Base filters | $128$ (up to 1024) | $64$ |
| Epochs | $30$ (paper: $100$) | $50$ |
| Batch size | $100$ | $100$ |
| KL scalar ($\beta$) | $0.05$ | $0.5$ |
| Seeds | 1–5 | 1–5 |

Both presets train in full float32, as in the paper. (`mixed_precision: True` in `config/local_config.py` trains
~2.5x faster on the T4, but CIFAR-10 seeds diverged to NaN within a few epochs, so it is off.)

*CIFAR-10 is trained for 30 instead of 100 epochs so the whole notebook finishes in ~4 h (results will be
somewhat worse than the paper). This is the long step: ~59 min per CIFAR-10 round of two seeds, i.e. ~3 h for all
10 runs (MNIST runs on the GPU left idle during CIFAR-10 seed 5, so it adds no time). Use
**Save Version → Save & Run All** and start from a fresh session. Finished seeds are skipped on re-run.*

In [ ]:
# Longest jobs first: CIFAR-10 seeds fill both GPUs, then the short MNIST seeds use the GPU
# that would otherwise sit idle while CIFAR-10 seed 5 trains.
# For a quick test run, use e.g. jobs = [(3, 1)]  (MNIST seed 1 only)
jobs = [(4, seed) for seed in range(1, 6)] + [(3, seed) for seed in range(1, 6)]

start_training = time.time()
run_jobs(jobs)
print(f"Total Training Time: {(time.time() - start_training) / 60:.2f} minutes")

## 6. Active Dimension Discovery & Sample Generation
Using the trained models:
1. Computes the **Jacobian-weighted sensitivity** ($\mathbf{w}_{\hat{\sigma}}$, Eq. 19) to filter out noise and prune collapsed axes.
2. Identifies the number of **ACTIVE** dimensions explaining 99% of variance.
3. Generates $10{,}000$ samples from the active latent dimensions for FID and Precision-Recall scoring.


In [ ]:
# Generate samples and discover active dimensions for MNIST (L=16)
%cd {REPO_DIR}/eval/study_fid/generate_samples
print("--- Evaluating Active Dimensions & Generating Samples for MNIST ---")
!python -u generate_samples.py --config_id 3 --latent_dim 16 --gen_type generation --eval_ids 1 2 3 4 5

# Generate samples and discover active dimensions for CIFAR-10 (L=128)
print("\n--- Evaluating Active Dimensions & Generating Samples for CIFAR-10 ---")
!python -u generate_samples.py --config_id 4 --latent_dim 128 --gen_type generation --eval_ids 1 2 3 4 5
%cd {REPO_DIR}



## 7. Compute FID, Precision, and Recall Scores
Evaluate Fréchet Inception Distance (FID) and k-NN Precision/Recall for all 5 seeds on the generated samples.


In [ ]:
%cd {REPO_DIR}/eval/study_fid/compute_fid

print("--- Computing FID Scores for MNIST (L=16) ---")
!python -u compute_n_plot_fid.py --config_id 3 --latent_dim 16 --gen_type generation --eval_ids 1 2 3 4 5

print("\n--- Computing FID Scores for CIFAR-10 (L=128) ---")
!python -u compute_n_plot_fid.py --config_id 4 --latent_dim 128 --gen_type generation --eval_ids 1 2 3 4 5

%cd {REPO_DIR}



## 8. Final Table 2 Reproduction
Parse the logged metrics across all 5 seeds and display the complete **Table 2** matching the paper!


In [ ]:
import numpy as np
import pandas as pd
import os

def _stat(path):
    if not os.path.exists(path):
        return "n/a"
    v = np.load(path)
    return f"{np.mean(v):.2f} ± {np.std(v):.2f}"

def load_dataset_results(dataset_name, latent_dim):
    gen_dir = f"eval/study_fid/generate_samples/logs/{dataset_name}/generation/Dim_{latent_dim}"
    fid_dir = f"eval/study_fid/compute_fid/logs/{dataset_name}/generation/Dim_{latent_dim}"
    return [_stat(f"{gen_dir}/rel_axis_stat.npy"), _stat(f"{fid_dir}/fid_scores_stat.npy"),
            _stat(f"{fid_dir}/precision_stat.npy"), _stat(f"{fid_dir}/recall_stat.npy")]

mnist = load_dataset_results("MNIST", 16)
cifar = load_dataset_results("CIFAR10", 128)

# Paper Table 2 rows: (MNIST active, FID, prec, rec, CIFAR-10 active, FID, prec, rec)
paper_rows = {
    "VAE":                      ["16", "28.78 ± 0.48", "0.88", "0.97", "128", "147.74 ± 0.81", "0.50", "0.47"],
    "β-TCVAE":                  ["16", "50.62 ± 1.19", "0.82", "0.95", "128", "180.94 ± 1.16", "0.30", "0.41"],
    "RAE":                      ["16", "18.79 ± 0.31", "0.87", "0.95", "128", "94.34 ± 1.58", "0.74", "0.47"],
    "WAE":                      ["16", "25.42 ± 1.19", "0.92", "0.92", "128", "140.49 ± 0.64", "0.42", "0.31"],
    "GECO-L0-ARM-VAE":          ["10.00 ± 1.10", "304.75 ± 64.29", "0.03", "0.38", "68.00 ± 2.97", "320.75 ± 45.09", "0.02", "0.04"],
    "MaskAAE":                  ["9.80 ± 1.60", "144.92 ± 16.80", "0.00", "0.07", "3.80 ± 0.40", "298.30 ± 8.44", "0.07", "0.04"],
    "ARD-VAE (Paper Reported)": ["12.80 ± 0.40", "22.24 ± 0.57", "0.91", "0.98", "105.80 ± 1.33", "87.56 ± 1.21", "0.82", "0.51"],
    "ARD-VAE (Reproduced Ours)": mnist + cifar,
}
columns = ["MNIST Active", "MNIST FID (↓)", "MNIST Prec (↑)", "MNIST Rec (↑)",
           "CIFAR-10 Active", "CIFAR-10 FID (↓)", "CIFAR-10 Prec (↑)", "CIFAR-10 Rec (↑)"]
df_table2 = pd.DataFrame.from_dict(paper_rows, orient="index", columns=columns)
df_table2.index.name = "Method"
print("\n================================== TABLE 2 REPRODUCTION SUMMARY ==================================")
display(df_table2)

## 9. Qualitative Visual Results
Display sample reconstructions and cumulative explained variance curves showing how ARD-VAE isolates the active latent dimensions.


In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image

def display_latest_image(pattern, title=""):
    files = glob.glob(pattern)
    if files:
        img_path = max(files, key=os.path.getmtime)
        img = Image.open(img_path)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14)
        plt.show()
    else:
        print(f"No image found matching: {pattern}")

print("--- Cumulative Explained Variance Curve (Relevance Profile) ---")
display_latest_image("eval/study_fid/generate_samples/logs/MNIST/generation/Dim_16/run_id_1/relevance_stat/Cum_explained_var.png", "MNIST Explained Variance per Latent Axis")
display_latest_image("eval/study_fid/generate_samples/logs/CIFAR10/generation/Dim_128/run_id_1/relevance_stat/Cum_explained_var.png", "CIFAR-10 Explained Variance per Latent Axis")

print("--- MNIST Reconstructions / Generated Samples (end of training) ---")
display_latest_image("logs/MNIST/Dim_16/Run_1/Output/Reconstructed_Images/*.png", "MNIST Reconstructions (input | reconstruction)")
display_latest_image("logs/MNIST/Dim_16/Run_1/Output/Generated_Images/*.png", "MNIST Generated Samples")

print("--- CIFAR-10 Reconstructions / Generated Samples (end of training) ---")
display_latest_image("logs/CIFAR10/Dim_128/Run_1/Output/Reconstructed_Images/*.png", "CIFAR-10 Reconstructions (input | reconstruction)")
display_latest_image("logs/CIFAR10/Dim_128/Run_1/Output/Generated_Images/*.png", "CIFAR-10 Generated Samples")